# NEOFC - Collect sample characteristics

In [1]:
from pathlib import Path 
import pandas as pd 
na = slice(None)

from utils import load_neofc_stats, load_sac_gc

# working directory
wd = Path.cwd() 
print(wd)

/Users/llotter/projects/mapfc


## Collect for each sample

In [2]:
# dict to store data
sample_dict = {}

# function to get final subject IDs from neofc stats
def get_subjects(sample, dset=None, index="run", remove_prefix=False, as_int=False, dx=None):
    df = load_neofc_stats(sample, parcs="Schaefer200", stats="auc", dset=dset, index_special=index, level="individual")
    if dx is not None:
        df = df.query(f"dx=='{dx}'")
    subs = df.index.get_level_values("id").unique().to_list()        
    if remove_prefix:
        subs = [s.split("-")[1] for s in subs]
    if as_int:
        subs = [int(s) for s in subs]
    return subs

# function to get global correlation
def get_gc(sample, parcs="Schaefer200", index="run", remove_prefix=False, as_int=False):
    df = load_sac_gc(sample, parcs=parcs, index_special=index)
    df = df.loc[parcs, ("pearson" if not "meg" in sample else "aec")]
    if "drug" in sample:
        df = df.query("treat=='placebo'")
    df = df["gc"].groupby("sub").mean()
    if remove_prefix:
        df.index = [s.split("-")[1] for s in df.index]
    if as_int:
        df.index = [int(s) for s in df.index]
    return df

## HCP - MRI

In [3]:
df = (
    # load
    pd.read_csv(wd / "data_deriv" / "pheno" / "hcp_ya" / "hcp_ya_pheno.csv", index_col=0)
    # restrict to final subjects
    .loc[get_subjects("hcp_ya_mri", "pet", remove_prefix=True, as_int=True)]
    # merge with QC
    .merge(
        pd.read_table(wd / "data_source" / "timeseries" / "hcp_ya_mri" / "linc_qc.tsv", index_col=0)
        .loc[:, ["mean_fd"]]
        .groupby("sub").mean(),
        left_index=True, right_index=True
    )
    # merge with gc
    .merge(get_gc("hcp_ya_mri", remove_prefix=True, as_int=True), left_index=True, right_index=True)
    # select variables
    .loc[:, ["sex", "age", "bmi", "mean_fd", "gc"]]
    # rename variables
    .rename(columns={"sex": "gender"})
)

# save in dict
print(f"n = {len(df)}")
sample_dict["HCP-YA: MRI"] = df

Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-..._dset-pet_stat-..._individual.csv.gz
n = 112


### HCP - MEG

In [4]:
# no QC data available

df = (
    # load
    pd.read_csv(wd / "data_source" / "pheno" / "hcp_ya" / "hcp_ya_pheno_open.csv", index_col=0)
    .merge(pd.read_csv(wd / "data_source" / "pheno" / "hcp_ya" / "hcp_ya_pheno_restricted.csv", index_col=0), 
           left_index=True, right_index=True)
    # restrict to final subjects
    .loc[get_subjects("hcp_ya_meg", index="fqband", remove_prefix=True, as_int=True)]
    # merge with gc
    .merge(get_gc("hcp_ya_meg", index="fqband", remove_prefix=True, as_int=True), left_index=True, right_index=True)
    # select variables
    .loc[:, ["Gender", "Age_in_Yrs", "gc"]]
    # rename variables
    .rename(columns={"Gender": "gender", "Age_in_Yrs": "age"})
)

# save in dict
print(f"n = {len(df)}")
sample_dict["HCP-YA: MEG"] = df

Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_meg/parc-..._stat-..._individual.csv.gz
n = 33


### YRSP

In [5]:
# no sample data available
# 27 subjects (26.52 ± 4.04 years old; 16/11 females/males; 25/2 right/left-handed)

df = (
    # nothing to load, so only load qc (FD)
    pd.read_table(wd / "data_source" / "timeseries" / "yrsp" / "linc_qc.tsv", index_col=0)
    .groupby("sub")[["mean_fd"]].mean()
    # select final subjects
    .loc[get_subjects("yrsp", remove_prefix=True)]
    # merge with gc
    .merge(get_gc("yrsp", remove_prefix=True), left_index=True, right_index=True)
)

# save in dict
print(f"n = {len(df)}")
sample_dict["YRSP"] = df

Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/yrsp/parc-..._stat-..._individual.csv.gz
n = 26


### Drug: MPH

In [6]:
# no sample data available
# no QC data available
# 30 subjects (33 ± 9.5 years old, 11/10 females/males)

df = (
    # nothing to load, so only load gc
    get_gc("drug_mph", index="treat")
    .to_frame()
)

# save in dict
sample_dict["MPH (placebo)"] = df

### Drug: Risperidone

In [7]:
# no age data available
# 21 subjects (27.56 ± 6.87 years old, all males)

df = (
    # load
    pd.read_table(wd / "data_source" / "timeseries" / "drug_risp" / "participants.tsv", index_col=0)
    .query("treatment=='placebo'")
    # select final subjects
    .loc[get_subjects("drug_risp", index="treat")]
    # merge with QC
    .merge(
        pd.read_table(wd / "data_source" / "timeseries" / "drug_risp" / "linc_qc.tsv")
        .assign(sub=lambda df: "sub-" + df["sub"].astype(str),
                ses=lambda df: "ses-" + df["ses"].astype(str))
        .query("space=='fsLR'"),
        left_on=["participant_id", "session_id"], right_on=["sub", "ses"], how="left", 
    )
    .set_index("sub")
    # merge with gc
    .merge(get_gc("drug_risp", index="treat"), left_index=True, right_index=True, how="left")
    # select variables
    .loc[:, ["sex", "mean_fd", "gc"]]
    # rename variables
    .rename(columns={"sex": "gender"})
)

print(f"n = {len(df)}")
sample_dict["Risperidone (placebo)"] = df

Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/drug_risp/parc-..._stat-..._individual.csv.gz
n = 19


### Drug: Ketamine/Midazolam

In [8]:
# no age data available
# 30 subjects (27.3 ± 6.2 years old, all males)

for dset, k in [("ket", "Ketamine (placebo)"), ("mdz", "Midazolam (placebo)")]:

    df = (
        # load
        pd.read_table(wd / "data_source" / "timeseries" / "drug_ket_mdz" / "participants.tsv", index_col=0)
        .query("treatment=='placebo'")
        # select final subjects
        .loc[get_subjects(f"drug_{dset}", index="treat")]
        # merge with QC
        .merge(
            pd.read_table(wd / "data_source" / "timeseries" / "drug_ket_mdz" / "linc_qc.tsv")
            .assign(sub=lambda df: "sub-" + df["sub"].astype(str).str.zfill(3),
                    ses=lambda df: "ses-" + df["ses"].astype(str))
            .query("space=='fsLR' and acq=='infusion'"),
            left_on=["participant_id", "session_id"], right_on=["sub", "ses"], how="left", 
        )
        .set_index("sub")
        # merge with gc
        .merge(get_gc(f"drug_{dset}", index="treat"), left_index=True, right_index=True, how="left")
        # select variables
        .loc[:, ["sex", "mean_fd", "gc"]]
        # rename variables
        .rename(columns={"sex": "gender"})
    )

    print(f"n = {len(df)}")
    sample_dict[k] = df

Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/drug_ket/parc-..._stat-..._individual.csv.gz
n = 29
Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/drug_mdz/parc-..._stat-..._individual.csv.gz
n = 28


### HCP-EP

In [9]:
for dx, k in [("CTRL", "HCP-EP (controls)"), ("SCZ", "HCP-EP (PSY)")]:
    
    df = (
        # load
        pd.read_csv(wd / "data_deriv" / "pheno" / "hcp_ep" / "hcpep_pheno.csv", index_col=0)
        # restrict to final subjects
        .loc[get_subjects("hcp_ep", index="dx", dx=dx)]
        # merge with QC
        .merge(
            pd.read_table(wd / "data_source" / "timeseries" / "hcp_ep" / "linc_qc.tsv")
            .assign(sub=lambda df: "sub-" + df["sub"].astype(str))
            .groupby("sub")[["mean_fd"]].mean(),
            left_index=True, right_index=True
        )
        # merge with gc
        .merge(get_gc("hcp_ep", index="dx"), left_index=True, right_index=True, how="left")
        # select variables
        .loc[:, ["sex", "age", "sestot", "panss_total", "panss_pos", "panss_neg", "apd_exp_months", 
                 "apd_chlor_equiv", "nih_totalcogcomp_unadjusted", "mean_fd", "gc"]]
        # rename variables
        .rename(columns={"sex": "gender"})
    )

    # save in dict
    print(f"n = {len(df)}")
    sample_dict[k] = df

Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/hcp_ep/parc-..._stat-..._individual.csv.gz
n = 55
Loading individual stats: /Users/llotter/projects/mapfc/results/neofc/hcp_ep/parc-..._stat-..._individual.csv.gz
n = 96


## Combine data

In [10]:
# all variables
variables = set()
for df in sample_dict.values():
    variables.update(df.columns)

# create empty dataframe to store sample characteristics
sample_df = pd.DataFrame(index=pd.Index(["n"] + list(variables), name="Variable"), columns=sample_dict.keys())


# go through samples:
for sample, df in sample_dict.items():
    
    # count
    sample_df.loc["n", sample] = len(df)
    
    # go through variables
    for var in variables:
    
        # if variables not not availble, skip
        if var not in df.columns or df[var].isna().all():
            continue
        
        # all numerical variables: report mean ± std (min - max in parentheses)
        if not var == "gender":
            sample_df.loc[var, sample] = f"{df[var].mean():.2f} ± {df[var].std():.2f} ({df[var].min():.2f} - {df[var].max():.2f})"
            
        # for non-numerical variables: report counts
        else:
            sample_df.loc[var, sample] = ", ".join([f"{c}={n}" for c, n in df[var].value_counts().items()])
            
# manual insertions
sample_df.loc["age", "YRSP"] = "26.52 ± 4.04 *"
sample_df.loc["gender", "YRSP"] = "F=16, M=11 *"
sample_df.loc["age", "MPH (placebo)"] = "33 ± 9.5 *"
sample_df.loc["gender", "MPH (placebo)"] = "F=11, M=10 *"
sample_df.loc["age", "Risperidone (placebo)"] = "27.56 ± 6.87 *"
sample_df.loc["gender", "Risperidone (placebo)"] = "F=0, M=21 *"
sample_df.loc["age", "Ketamine (placebo)"] = "27.3 ± 6.2 *"
sample_df.loc["gender", "Ketamine (placebo)"] = "F=0, M=30 *"
sample_df.loc["age", "Midazolam (placebo)"] = "27.3 ± 6.2 *"    
sample_df.loc["gender", "Midazolam (placebo)"] = "F=0, M=30 *"
        
# sort
sample_df = sample_df.loc[["n", "gender", "age", "mean_fd", "gc", "bmi", 
                           "sestot", "panss_total", "panss_pos", "panss_neg", 
                           "apd_exp_months", "apd_chlor_equiv", "nih_totalcogcomp_unadjusted"]]

# save
sample_df.to_csv(wd / "data_deriv" / "pheno" / "sample_characteristics.csv")

sample_df

,HCP-YA: MRI,HCP-YA: MEG,YRSP,MPH (placebo),Risperidone (placebo),Ketamine (placebo),Midazolam (placebo),HCP-EP (controls),HCP-EP (PSY)
Variable,,,,,,,,,
n,112,33,26,30,19,29,28,55,96
gender,"F=64, M=48","M=17, F=16","F=16, M=11 *","F=11, M=10 *","F=0, M=21 *","F=0, M=30 *","F=0, M=30 *","M=36, F=19","M=56, F=40"
age,29.01 ± 3.65 (22.00 - 36.00),28.55 ± 3.54 (22.00 - 35.00),26.52 ± 4.04 *,33 ± 9.5 *,27.56 ± 6.87 *,27.3 ± 6.2 *,27.3 ± 6.2 *,24.82 ± 4.07 (16.83 - 35.67),22.66 ± 3.62 (16.67 - 34.83)
mean_fd,0.15 ± 0.04 (0.09 - 0.26),NaN,0.15 ± 0.04 (0.10 - 0.25),NaN,0.19 ± 0.07 (0.07 - 0.36),0.15 ± 0.05 (0.08 - 0.31),0.15 ± 0.05 (0.08 - 0.29),0.11 ± 0.04 (0.05 - 0.21),0.13 ± 0.05 (0.04 - 0.26)
gc,0.02 ± 0.01 (0.01 - 0.04),0.11 ± 0.03 (0.05 - 0.21),0.02 ± 0.01 (0.01 - 0.03),0.04 ± 0.02 (0.01 - 0.11),0.06 ± 0.03 (0.02 - 0.15),0.10 ± 0.07 (0.02 - 0.24),0.10 ± 0.07 (0.02 - 0.24),0.02 ± 0.01 (0.01 - 0.04),0.02 ± 0.01 (0.01 - 0.04)
bmi,26.03 ± 4.86 (18.44 - 39.60),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sestot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.06 ± 1.07 (1.00 - 5.00),2.21 ± 1.17 (1.00 - 5.00)
panss_total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.19 ± 11.32 (29.00 - 78.00)
panss_pos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.09 ± 3.99 (7.00 - 23.00)
